In [ ]:
r"""
CS781 Phase 3 - Rhetorical Role Prediction (RR): KNOWLEDGE DISTILLATION  Model 3 (teacher)  ->  Model 4 (student)
==================================================================================================================

Teacher : Model 3 (decoder-only, RoPE + GQA + SwiGLU, 3-head MTP, d=640, 8 layers)  + LoRA + sentence head
Student : Model 4 (MSM: 3 x [Local-128 -> Local-256 -> Memory-Global], d=512, 9 layers)  + LoRMA-plus + sentence head
          (STUDENT_PEFT = "lorma" | "lora" | "full").  Both models use the SAME 50K word-level BPE (LegalBPE, copied verbatim
          from the Phase-1 notebooks: lower-cased words/digits only, punctuation is invisible to the models).

Pipeline (one run, every stage resumable - see "resuming"):
  A. TEACHER   fine-tune Model 3 + LoRA + head on the gold labels (class-weighted CE, rare-label oversampling, dev early stopping).
               Then the best teacher produces SOFT LABELS (logits) for every sentence of train / dev / test and saves them.
  B. STUDENT   fine-tune Model 4 + LoRMA + head with the distillation loss
                   L = sum_i w_i * [ alpha * CE(s_i, y_i)  +  (1 - alpha) * T^2 * KL( softmax(t_i / T) || softmax(s_i / T) ) ] / sum_i w_i
               w_i = capped inverse-sqrt class weight of the gold label of sentence i in its domain (same imbalance handling as
               before, applied to BOTH terms so rare roles keep their weight in the soft-label term too).
               Dev early stopping, then post-hoc calibration (tau, unseen-label mask) exactly as in the prefix notebook.
  C. REPORT    dev Macro-F1 of teacher / student / teacher+student ensemble, and the submission of SUBMIT_MODEL.

Because both models read the same tokens, the teacher's soft labels are indexed by (document, sentence) and simply looked up
while the student trains on its own random windows - the teacher is never run again after stage A.

Data / checkpoints / tokenizer are located exactly as in phase3-model3.ipynb: Phase 3 json (attached copy or HF repo, Kaggle
secret "HF"), best.pt + best_meta.json (tag-checked) of each model under /kaggle/input, vocab_50k.json + merges_50k.json.
Override with env: MODEL3_CKPT, MODEL4_CKPT, BPE_DIR, PHASE3_DIR.   SMOKE=1: tiny end-to-end check (no submission).

Resuming (usage limits!): stage A writes  teacher_best.pt  and  teacher_soft.pt  into WORK_DIR.  Attach that notebook output as an
input of the next session and stage A is skipped automatically (teacher_soft.pt found) - only the student trains.

Outputs (WORK_DIR): data_report.json, teacher_best.pt, teacher_soft.pt, student_best.pt, kd_report.json, kd_ckpt/*.csv,
                    prediction.json + submission.zip (the SUBMIT_MODEL), prediction_{teacher,student,ensemble}.json
"""
import os
import re
import csv
import json
import math
import time
import hashlib
import zipfile
import random
from pathlib import Path
from dataclasses import dataclass, fields
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# CONFIG
# ============================================================================
SEED = 1234
SMOKE = os.environ.get("SMOKE", "0") == "1"      # tiny end-to-end run (2+2 train docs, 1 epoch per stage, no submission)

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = ("/kaggle/working/phase3_rr_kd" if IS_KAGGLE else "./phase3_rr_kd") + ("_smoke" if SMOKE else "")
CKPT_DIR = os.path.join(WORK_DIR, "kd_ckpt")
INPUT_ROOT = "/kaggle/input"

# ---- data (same repo / file names as phase3.ipynb) ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "label_vocab": "Phase_3_RR/Train/label_vocab.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json",
    "IT_test": "Phase_3_RR/Test/IT_test.json",
}
LOCAL_PHASE3_DIR = os.environ.get("PHASE3_DIR", "")      # optional: root that contains Phase_3_RR/

# ---- the two base models (checkpoints are tag-checked) ----
TEACHER_TAG = "decoder_only_rope_gqa_swiglu_mtp4_v1"            # Model 3
STUDENT_TAG = "decoder_only_msm_rope_gqa_swiglu_mtp3_v2"        # Model 4
TEACHER_CKPT_PATH = os.environ.get("MODEL3_CKPT", "")           # explicit best.pt / latest.pt; empty = search /kaggle/input
STUDENT_CKPT_PATH = os.environ.get("MODEL4_CKPT", "")
BPE_DIR = os.environ.get("BPE_DIR", "")                         # folder with vocab_50k.json + merges_50k.json; empty = search
FORCE_TOKENIZER = os.environ.get("FORCE_TOKENIZER", "0") == "1" # turn the tokenizer/checkpoint sanity error into a warning

# ---- windowing (tokens) - identical for teacher and student (same tokens) ----
MAX_LEN = 1024               # Model 4's memory is built for 1024 tokens (8 chunks x 128); Model 3 allows 1088
MAX_SENT_TOK = 160
SENT_PREFIX = ""
SENT_END = "none"            # "none" | "eos"
TARGET_TOK = 320             # tokens of target sentences per window at inference
TRAIN_TARGET_RANGE = (160, 480)

# ---- sentence head (same for both models; width = the model's d_model) ----
POOLING = "last_mean"        # "last" | "last_mean" (last token + mean over the sentence's tokens)
USE_POS_FEATURE = True
N_POS_BUCKETS = 20
USE_DOMAIN_FEATURE = True
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.2

# ---- teacher: Model 3 + LoRA ----
T_LORA_R = 32
T_LORA_ALPHA = 64
T_LORA_DROPOUT = 0.05
T_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
T_TRAIN_HEAD0 = True         # also train MTP head 0 (640x640 projection + RMSNorm) - the layer that produces the sentence features
T_FT_DROPOUT = 0.1
EPOCHS_TEACHER = 15
LR_T_ADAPTER = 3e-4
LR_T_BACKBONE = 1e-4

# ---- student: Model 4 + a DIFFERENT PEFT method ----
STUDENT_PEFT = "lorma"       # "lorma" (multiplicative low-rank, LoRMA+) | "lora" | "full" (all blocks; embeddings stay frozen)
S_R = 32
S_ALPHA = 32                 # LoRMA-plus scaling = alpha / r
S_DROPOUT = 0.05
S_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down",      # every projection of the local blocks and
             "wq_proj", "wk_proj", "wv_proj", "write_out", "gate_proj")                 # the memory blocks' write / gate paths
S_TRAIN_FINAL_NORM = True
S_FT_DROPOUT = 0.10          # Model 4 pretraining FFN dropout
EPOCHS_STUDENT = 20
LR_S_ADAPTER = 4e-4          # LoRMA paper range 3e-4 .. 5e-4 (use ~5e-4 for "lora", ~1e-4 for "full")
LR_S_BACKBONE = 1e-4         # final norm / "full" fine-tuning

# ---- knowledge distillation ----
KD_ALPHA = 0.5               # weight of the gold-label CE; (1 - alpha) goes to the soft-label term
KD_TEMP = 2.0                # temperature T (soft term is multiplied by T^2)
SUBMIT_MODEL = "student"     # which predictions become prediction.json / submission.zip: "student" | "teacher" | "ensemble"

# ---- optimisation (both stages) ----
MICRO_BS = 4
ACCUM = 2
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
PATIENCE = 5                 # stop a stage when dev Macro-F1 has not improved for this many epochs
# ---- class imbalance handling (unchanged) ----
DOMAINS = ["CL", "IT"]
CLASS_WEIGHT_POWER = 0.5
CLASS_WEIGHT_CLIP = (0.25, 8.0)
PER_DOMAIN_WEIGHTS = True
OVERSAMPLE_RARE = True
RARE_FRAC = 0.01
RARE_REPEAT = 3
POSTHOC_CALIBRATION = True
TAU_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
LABEL_SMOOTHING = 0.0
CE_CHUNK = 2048              # (Model 4's MTP training loss helper; unused here)
CE_RECOMPUTE = False

AMP = {"enabled": False, "dtype": torch.float16}


# ============================================================================
# 1. DATA LOADING  (as in phase3-model3 / phase3.ipynb)
# ============================================================================
def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ============================================================================
# 1. DATA LOADING  (same layout handling as phase3.ipynb, token via Kaggle secret - never hard-coded)
# ============================================================================
def get_hf_token():
    """never hard-code a token in a notebook that can be shared: Kaggle secret 'HF' (Add-ons > Secrets) or env HF_TOKEN"""
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or None


def resolve_phase3_files():
    paths = {}
    roots = []
    if LOCAL_PHASE3_DIR and Path(LOCAL_PHASE3_DIR).is_dir():
        roots.append(Path(LOCAL_PHASE3_DIR))
    roots += [Path(INPUT_ROOT), Path(WORK_DIR), Path.cwd()]
    wanted = {Path(p).name: k for k, p in PHASE3_FILES.items()}
    for root in roots:
        if not root.exists():
            continue
        for fp in root.rglob("*.json"):
            if fp.name in wanted:
                paths.setdefault(wanted[fp.name], str(fp))
    missing = [k for k in PHASE3_FILES if k not in paths]
    if missing:
        from huggingface_hub import hf_hub_download
        token = get_hf_token()
        for k in missing:
            log("downloading", PHASE3_FILES[k])
            paths[k] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE3_FILES[k],
                                       repo_type="dataset", token=token)
    return paths


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)



def normalize_documents(raw, split_name):
    """Normalize Phase 3 RR JSON into:
       [{"id": ..., "text": [...], "labels": [...] or None}]
    """
    docs = []

    def is_sentence_list(value):
        return (
            isinstance(value, list)
            and all(isinstance(s, str) for s in value)
        )

    def visit(obj, inherited_id=None):
        if isinstance(obj, list):
            # A list of strings may be a document's sentence list.
            if is_sentence_list(obj):
                docs.append({
                    "id": str(inherited_id or f"{split_name}_{len(docs)}"),
                    "text": obj,
                    "labels": None,
                })
                return

            for item in obj:
                visit(item, inherited_id)
            return

        if not isinstance(obj, dict):
            # Ignore metadata strings and other scalar values.
            return

        text = obj.get("text", obj.get("sentences"))
        labels = obj.get("labels", obj.get("label"))

        # Some files store sentences under a document field.
        if text is None:
            for key in ("data", "documents", "docs", "items"):
                if key in obj:
                    visit(obj[key], inherited_id or obj.get("id"))
                    return

            # Wrapper dictionaries may contain documents under arbitrary keys.
            for key, value in obj.items():
                if isinstance(value, (dict, list)):
                    visit(value, str(key))
            return

        if isinstance(text, str):
            text = [text]

        if not isinstance(text, list) or not all(
            isinstance(s, str) for s in text
        ):
            raise TypeError(
                f"{split_name}: invalid text format for document "
                f"{obj.get('id', inherited_id)}"
            )

        # Normalize labels only when present and valid.
        if isinstance(labels, list):
            labels = [str(x) for x in labels]
        else:
            labels = None

        docs.append({
            "id": str(obj.get("id", inherited_id or
                              f"{split_name}_{len(docs)}")),
            "text": text,
            "labels": labels,
        })

    visit(raw)

    if not docs:
        raise ValueError(f"{split_name}: no documents found")

    # Check IDs and label alignment.
    ids = [d["id"] for d in docs]
    if len(ids) != len(set(ids)):
        raise ValueError(f"{split_name}: duplicate document IDs detected")

    for d in docs:
        if d["labels"] is not None and len(d["labels"]) != len(d["text"]):
            raise ValueError(
                f"{split_name}: label/text length mismatch in {d['id']}: "
                f"{len(d['labels'])} labels vs {len(d['text'])} sentences"
            )

    return docs



def load_labels(raw):
    if isinstance(raw, dict) and "labels" in raw:
        raw = raw["labels"]
    if isinstance(raw, dict):
        raw = [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    labels = [str(x) for x in raw]
    assert len(labels) == len(set(labels))
    return labels


def load_phase3():
    paths = resolve_phase3_files()
    raw = {k: read_json(p) for k, p in paths.items()}
    labels = load_labels(raw["label_vocab"])
    data = {k: normalize_documents(raw[k], k) for k in raw if k != "label_vocab"}
    l2i = {l: i for i, l in enumerate(labels)}
    for name, docs in data.items():
        labelled = name.endswith("_train") or name.endswith("_dev")
        for d in docs:
            d["domain"] = name.split("_")[0]
            if labelled:
                assert d["labels"] is not None and len(d["labels"]) == len(d["text"]), (name, d["id"])
                bad = [x for x in d["labels"] if x not in l2i]
                assert not bad, f"{name}/{d['id']}: unknown labels {set(bad)}"
    return data, labels, l2i


# ============================================================================
# 2a. MODEL 3 (teacher) ARCHITECTURE  (as in phase3-model3; names suffixed with 3 so both models can live in one file)
# ============================================================================
@dataclass
class ModelConfig3:
    vocab_size: int = 50_048          # embedding rows (50,000 real ids + padding rows)
    real_vocab: int = 50_000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 1536
    dropout: float = 0.1
    max_len: int = 1088
    mtp_heads: int = 3
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    bos_id: int = 3
    mask_id: int = 4


def config_from_ckpt3(ck):
    """the checkpoint stores asdict(ModelConfig3) under 'config'; unknown keys (mtp_weights ...) are ignored"""
    names = {f.name for f in fields(ModelConfig3)}
    return ModelConfig3(**{k: v for k, v in (ck.get("config") or {}).items() if k in names})


class RMSNorm3(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.float().pow(2).mean(-1, keepdim=True)
        inv_rms = torch.rsqrt(variance + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)


def build_rope3(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def apply_rope3(x, cos, sin):
    cos = cos.to(dtype=x.dtype)
    sin = sin.to(dtype=x.dtype)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


def repeat_kv3(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd); query head h reads kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention3(nn.Module):
    """causal grouped-query attention with RoPE (10 query heads share 2 key/value heads) + optional learned K/V prefix"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)
        self._capture = None             # list -> forward() stores this layer's (pre-RoPE K, V) (used to initialise the prefix)

    def forward(self, x, cos, sin, prefix=None, cos_p=None, sin_p=None, mask=None):
        """cos/sin: RoPE tables of the REAL tokens' positions.  prefix = (K, V), each (n_kv, P, hd), K is pre-RoPE and gets
        the positions 0..P-1 (cos_p/sin_p); mask (T, P+T) bool: every real token sees all prefix slots + causal real tokens"""
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        if self._capture is not None:
            self._capture.append((k.detach().float(), v.detach().float()))
        q, k = apply_rope3(q, cos, sin), apply_rope3(k, cos, sin)
        if prefix is None:
            k, v = repeat_kv3(k, self.groups), repeat_kv3(v, self.groups)
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            pk, pv = prefix
            pk = apply_rope3(pk[None].to(k.dtype), cos_p, sin_p).expand(B, -1, -1, -1)
            pv = pv[None].to(v.dtype).expand(B, -1, -1, -1)
            k = repeat_kv3(torch.cat([pk, k], dim=2), self.groups)
            v = repeat_kv3(torch.cat([pv, v], dim=2), self.groups)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm3(cfg.d_model)
        self.attn = GQAttention3(cfg)
        self.norm2 = RMSNorm3(cfg.d_model)
        self.ffn = SwiGLU3(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, prefix=None, cos_p=None, sin_p=None, mask=None):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin, prefix, cos_p, sin_p, mask))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class MTPHead3(nn.Module):
    """z = RMSNorm3(x + W x): Model 3 has no separate final norm, head 0 (the next-token head) plays that role"""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm = RMSNorm3(dim)

    def forward(self, x):
        return self.norm(x + self.proj(x))


class DecoderLM3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block3(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([MTPHead3(cfg.d_model)])          # only head 0 (heads 1.. predict t+2.. and are not needed)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope3(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def hidden(self, idx, prefix=None):
        """idx (B, T) -> features z of MTP head 0, (B, T, D).
        prefix: None or (n_layers, 2, n_kv, P, hd) learned K/V.  With a prefix the real tokens sit at RoPE positions P..P+T-1."""
        B, T = idx.shape
        P = 0 if prefix is None else prefix.size(3)
        assert P + T <= self.cfg.max_len, f"prefix {P} + sequence {T} > max_len {self.cfg.max_len}"
        cos, sin = self.rope_cos[:P + T], self.rope_sin[:P + T]
        cos_q, sin_q = cos[P:][None, None], sin[P:][None, None]
        cos_p = sin_p = mask = None
        if P:
            cos_p, sin_p = cos[:P][None, None], sin[:P][None, None]
            mask = torch.cat([torch.ones(T, P, dtype=torch.bool, device=idx.device),
                              torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))], dim=1)
        x = self.drop(self.embed(idx))
        for i, blk in enumerate(self.blocks):
            pkv = None if prefix is None else (prefix[i, 0], prefix[i, 1])
            x = blk(x, cos_q, sin_q, pkv, cos_p, sin_p, mask)
        return self.heads[0](x)

    @torch.no_grad()
    def capture_kv(self, idx):
        """run the frozen model on idx (1, P) and return its real per-layer (pre-RoPE K, V): (n_layers, 2, n_kv, P, hd)"""
        for blk in self.blocks:
            blk.attn._capture = []
        try:
            self.hidden(idx)
            kv = [torch.stack([c[0][0], c[1][0]]) for c in (blk.attn._capture[0] for blk in self.blocks)]
        finally:
            for blk in self.blocks:
                blk.attn._capture = None
        return torch.stack(kv)


# ============================================================================
# 2b. MODEL 4 (student) ARCHITECTURE  (verbatim from model4.ipynb - same module / parameter names, so best.pt loads strictly;
#     names suffixed with 4.  Only trunk() is used here: the KV-cache generation path and the MTP loss stay unused.)
# ============================================================================
from torch.utils.checkpoint import checkpoint as torch_checkpoint

LOCAL_IMPL = {"mode": "block"}     # "block" = block-wise sliding window (fast); run_self_checks switches to "mask" if they disagree
_BLOCK_MASKS = {}


@dataclass
class ModelConfig4:
    vocab_size: int = 50_000
    d_model: int = 512
    n_heads: int = 8
    n_kv_heads: int = 2
    ffn_hidden: int = 1792
    attn_dropout: float = 0.0
    ffn_dropout: float = 0.10
    layer_types: tuple = ("local_small", "local_large", "memory") * 3
    window_small: int = 128
    window_large: int = 256
    n_mem_chunks: int = 8
    chunk_size: int = 128
    mem_per_chunk: int = 4
    max_len: int = 1088
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    n_mtp: int = 3
    mtp_weights: tuple = (1.0, 0.5, 0.25)
    mtp_pos_frac: float = 0.25

    @property
    def n_layers(self):
        return len(self.layer_types)


def config_from_ckpt4(ck):
    """the checkpoint stores asdict(ModelConfig) under 'config'"""
    names = {f.name for f in fields(ModelConfig4)}
    c = {k: v for k, v in (ck.get("config") or {}).items() if k in names}
    for k in ("layer_types", "mtp_weights"):
        if k in c:
            c[k] = tuple(c[k])
    return ModelConfig4(**c)


class RMSNorm4(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope4(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope4(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (1 or B, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def repeat_kv4(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd); query head h reads kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


def block_local_mask(window, n_blocks, device):
    """(n_blocks, 1, W, 2W) bool. A query i of a block looks at the 2W keys [previous block | own block]; key j is allowed iff
    i < j <= i + W, i.e. the token itself and the W-1 tokens before it. The first block has no previous block."""
    key = (window, n_blocks, str(device))
    m = _BLOCK_MASKS.get(key)
    if m is None:
        i = torch.arange(window, device=device)[:, None]
        j = torch.arange(2 * window, device=device)[None, :]
        base = (j > i) & (j <= i + window)                   # (W, 2W)
        m = base[None].repeat(n_blocks, 1, 1)                # (nb, W, 2W)
        m[0] &= (j >= window)                                # no previous block in front of block 0
        m = m[:, None]
        _BLOCK_MASKS[key] = m
    return m


def local_attention(q, k, v, window, dropout_p=0.0):
    """Exact causal sliding-window attention (a token sees itself and the window-1 tokens before it), computed block-wise:
    the queries of block n only meet the keys of blocks n-1 and n (2*window keys) instead of all T keys.
    q, k, v: (B, H, T, hd), T arbitrary (padded up to a multiple of window internally)."""
    B, H, T, hd = q.shape
    nb = (T + window - 1) // window
    Tp = nb * window
    if Tp != T:                          # padded keys sit after every real query, so causality hides them
        q, k, v = (F.pad(t, (0, 0, 0, Tp - T)) for t in (q, k, v))
    qb = q.reshape(B, H, nb, window, hd).transpose(1, 2).reshape(B * nb, H, window, hd)

    def two_blocks(t):                   # (B, H, Tp, hd) -> (B*nb, H, 2W, hd) = [previous block | own block]
        tb = t.reshape(B, H, nb, window, hd)
        prev = torch.cat([tb.new_zeros(B, H, 1, window, hd), tb[:, :, :-1]], dim=2)
        return torch.cat([prev, tb], dim=3).transpose(1, 2).reshape(B * nb, H, 2 * window, hd)

    mask = block_local_mask(window, nb, q.device)                                    # (nb, 1, W, 2W)
    mask = mask[None].expand(B, nb, 1, window, 2 * window).reshape(B * nb, 1, window, 2 * window)
    out = F.scaled_dot_product_attention(qb, two_blocks(k), two_blocks(v), attn_mask=mask, dropout_p=dropout_p)
    return out.reshape(B, nb, H, window, hd).transpose(1, 2).reshape(B, H, Tp, hd)[:, :, :T]


def local_attention_ref(q, k, v, window, dropout_p=0.0):
    """plain masked SDPA over all T keys - the slow reference (and the fallback) of local_attention"""
    T = q.size(2)
    i = torch.arange(T, device=q.device)
    mask = (i[None, :] <= i[:, None]) & (i[None, :] > i[:, None] - window)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout_p)


class GQAttention4(nn.Module):
    """causal grouped-query self-attention with RoPE, restricted to a sliding window (10 query heads share 2 key/value heads)"""

    def __init__(self, cfg, window):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.window = window
        self.p = cfg.attn_dropout
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope4(q, cos, sin)
        k = apply_rope4(k, cos, sin)
        k, v = repeat_kv4(k, self.groups), repeat_kv4(v, self.groups)
        p = self.p if self.training else 0.0
        attend = local_attention if LOCAL_IMPL["mode"] == "block" else local_attention_ref
        out = attend(q, k, v, self.window, p)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

    def forward_cached(self, x, cos, sin, k_cache, v_cache, start, attn_mask, pos_ids, kpos):
        """T new tokens go into cache slots [start, start+T); they attend to slots [0, start+T). attn_mask carries causality +
        validity in slot space; the window is applied on the REAL positions (pos_ids of the queries, kpos of every cached slot),
        because left-padded rows have different slot -> position offsets."""
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope4(q, cos, sin)
        k = apply_rope4(k, cos, sin)
        k_cache[:, :, start:start + T] = k.to(k_cache.dtype)
        v_cache[:, :, start:start + T] = v.to(v_cache.dtype)
        keys = repeat_kv4(k_cache[:, :, :start + T], self.groups).to(q.dtype)
        vals = repeat_kv4(v_cache[:, :, :start + T], self.groups).to(q.dtype)
        in_window = kpos[:, None, None, :] > (pos_ids[:, None, :, None] - self.window)       # (B, 1, T, S)
        out = F.scaled_dot_product_attention(q, keys, vals, attn_mask=attn_mask & in_window)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class LocalBlock(nn.Module):
    """pre-norm block: sliding-window GQA attention (RoPE) + SwiGLU4"""
    kind = "local"

    def __init__(self, cfg, window):
        super().__init__()
        self.norm1 = RMSNorm4(cfg.d_model)
        self.attn = GQAttention4(cfg, window)
        self.norm2 = RMSNorm4(cfg.d_model)
        self.ffn = SwiGLU4(cfg)
        self.drop = nn.Dropout(cfg.ffn_dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

    def forward_cached(self, x, cos, sin, cache, start, attn_mask, pos_ids, kpos):
        x = x + self.attn.forward_cached(self.norm1(x), cos, sin, cache["k"], cache["v"], start, attn_mask, pos_ids, kpos)
        x = x + self.ffn(self.norm2(x))
        return x


def memory_masks(q_chunk, k_chunk, k_valid, mem_per_chunk, n_slots):
    """Boolean masks of the memory layers, all slots / all rows at once.
        q_chunk (Bq, Tq)        chunk id of every query token
        k_chunk (Bk, S), k_valid (Bk, S)   chunk id / real-token flag of every key token (cached slots)
    returns
        w   (B, 1, slots, S)    WRITE: the memory tokens of chunk c may read exactly the (real) tokens of chunk c
        r   (B, 1, Tq, slots)   READ : a token of chunk j may read the memory tokens of the chunks < j  (strictly earlier = causal)
        has (B, Tq)             False for tokens with nothing to read (first chunk, padding rows): their memory output is zeroed
    Rows that would be empty (slots of chunks that have no token yet, tokens of chunk 0) get a harmless all-allowed fallback so
    that no softmax row is empty (NaN); the fallback slots are never read by a token, and the zeroed rows contribute nothing."""
    slot_chunk = torch.arange(n_slots, device=q_chunk.device) // mem_per_chunk                 # (slots,)
    w = (k_chunk[:, None, :] == slot_chunk[None, :, None]) & k_valid[:, None, :]               # (Bk, slots, S)
    w = torch.where(w.any(-1, keepdim=True), w, k_valid[:, None, :])
    r = slot_chunk[None, None, :] < q_chunk[:, :, None]                                        # (Bq, Tq, slots)
    has = r.any(-1)
    r = torch.where(has[..., None], r, torch.ones_like(r))
    return w[:, None], r[:, None], has


class MemoryBlock(nn.Module):
    """Memory-Global layer. WRITE (memory tokens <- the tokens of their own chunk, gated update of the memory carried over from
    the previous stage), READ (tokens <- memory tokens of strictly earlier chunks, GQA, no RoPE), then SwiGLU4. Returns (x, memory)."""
    kind = "memory"

    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = d // self.h
        self.groups = self.h // self.kv
        self.p = cfg.attn_dropout
        self.norm1 = RMSNorm4(d)
        self.norm2 = RMSNorm4(d)
        self.mem_norm_w = RMSNorm4(d)                 # norm of the memory as query of the write attention
        self.mem_norm_r = RMSNorm4(d)                 # norm of the memory as key/value of the read attention
        # write: memory tokens (queries) <- chunk tokens (keys / values), no RoPE
        self.wq_proj = nn.Linear(d, self.h * self.hd, bias=False)
        self.wk_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.wv_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.write_out = nn.Linear(self.h * self.hd, d, bias=False)
        self.gate_proj = nn.Linear(2 * d, d, bias=False)            # g = sigmoid(W [M_prev ; candidate])
        # read: tokens (queries) <- memory tokens (keys / values), no RoPE
        self.q_proj = nn.Linear(d, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, d, bias=False)
        self.ffn = SwiGLU4(cfg)
        self.drop = nn.Dropout(cfg.ffn_dropout)

    def write_kv(self, xn):
        B, T, _ = xn.shape
        k = self.wk_proj(xn).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv_proj(xn).view(B, T, self.kv, self.hd).transpose(1, 2)
        return k, v

    def update(self, mem_prev, k_all, v_all, w_mask):
        """gated memory update. mem_prev (B, slots, D); k_all / v_all (B, KV, S, hd) of every token seen so far"""
        B, n, _ = mem_prev.shape
        q = self.wq_proj(self.mem_norm_w(mem_prev)).view(B, n, self.h, self.hd).transpose(1, 2)
        keys = repeat_kv4(k_all, self.groups).to(q.dtype)
        vals = repeat_kv4(v_all, self.groups).to(q.dtype)
        out = F.scaled_dot_product_attention(q, keys, vals, attn_mask=w_mask, dropout_p=self.p if self.training else 0.0)
        cand = self.write_out(out.transpose(1, 2).reshape(B, n, -1)).type_as(mem_prev)
        gate = torch.sigmoid(self.gate_proj(torch.cat([mem_prev, cand], dim=-1)))
        return mem_prev + gate * (cand - mem_prev)                    # = (1 - g) * M_prev + g * candidate

    def read(self, xn, mem, r_mask, has):
        B, T, _ = xn.shape
        n = mem.size(1)
        mn = self.mem_norm_r(mem)
        q = self.q_proj(xn).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(mn).view(B, n, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(mn).view(B, n, self.kv, self.hd).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, repeat_kv4(k, self.groups), repeat_kv4(v, self.groups), attn_mask=r_mask,
                                             dropout_p=self.p if self.training else 0.0)
        out = self.o_proj(out.transpose(1, 2).reshape(B, T, -1))
        return out * has[..., None].to(out.dtype)

    def forward(self, x, mem_prev, mctx):
        xn = self.norm1(x)
        k, v = self.write_kv(xn)
        mem = self.update(mem_prev, k, v, mctx["w"])
        x = x + self.read(xn, mem, mctx["r"], mctx["has"])
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x, mem

    def forward_cached(self, x, mem_prev, cache, start, mctx):
        """the write keys/values of every token live in the cache (no RoPE), the memory is recomputed from them on every call;
        only completed chunks are readable, so this equals the full forward pass exactly"""
        xn = self.norm1(x)
        k, v = self.write_kv(xn)
        T = x.size(1)
        S = start + T
        cache["k"][:, :, start:S] = k.to(cache["k"].dtype)
        cache["v"][:, :, start:S] = v.to(cache["v"].dtype)
        mem = self.update(mem_prev, cache["k"][:, :, :S], cache["v"][:, :, :S], mctx["w"])
        x = x + self.read(xn, mem, mctx["r"], mctx["has"])
        x = x + self.ffn(self.norm2(x))
        return x, mem


def _ce_sum(h, E, t, pad_id):
    logits = F.linear(h, E)                                  # (chunk, V), fp16 under autocast
    return F.cross_entropy(logits.float(), t, ignore_index=pad_id, reduction="sum")


def ce_sum_count(h2d, E, tgt, pad_id, chunk=CE_CHUNK, ckpt=False):
    """summed cross-entropy of h2d @ E^T against tgt, computed chunk by chunk. With ckpt=True every chunk is recomputed in the
    backward pass instead of stored, so four heads never hold four sets of full logits at the same time."""
    total = h2d.new_zeros((), dtype=torch.float32)
    for i in range(0, h2d.size(0), chunk):
        hh, tt = h2d[i:i + chunk], tgt[i:i + chunk]
        if ckpt and torch.is_grad_enabled():
            total = total + torch_checkpoint(_ce_sum, hh, E, tt, pad_id, use_reentrant=False)
        else:
            total = total + _ce_sum(hh, E, tt, pad_id)
    return total, (tgt != pad_id).sum()


class DecoderLM4(nn.Module):
    """MSM transformer: 3 x [Local-128 -> Local-256 -> Memory-Global] with RoPE + GQA + SwiGLU4 and multi-token prediction.
    All heads read the same final hidden state h and share the tied embedding E as output matrix: head k (k = 1..K) predicts
    token t+k. Head 1 is the plain LM head (RMSNorm4(h) @ E^T) and the only one used for validation and decoding; heads 2..K are
    training-time auxiliaries  RMSNorm_k(h + W_k h) @ E^T.
    Memory: mem_init (32 x D) is the learned latent memory. The first memory layer updates it from the tokens, the result is
    handed to the next memory layer (M0 -> M1 -> M2). Memory slot c always belongs to chunk c = position // 128."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_slots = cfg.n_mem_chunks * cfg.mem_per_chunk
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection
        windows = {"local_small": cfg.window_small, "local_large": cfg.window_large}
        self.blocks = nn.ModuleList([MemoryBlock(cfg) if kind == "memory" else LocalBlock(cfg, windows[kind])
                                     for kind in cfg.layer_types])
        self.final_norm = RMSNorm4(cfg.d_model)
        self.mem_init = nn.Parameter(torch.empty(self.n_slots, cfg.d_model))      # learned latent memory
        self.mtp_proj = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False) for _ in range(cfg.n_mtp - 1)])
        self.mtp_norm = nn.ModuleList([RMSNorm4(cfg.d_model) for _ in range(cfg.n_mtp - 1)])
        cos, sin = build_rope4(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        nn.init.normal_(self.mem_init, mean=0.0, std=0.02)
        # damp the residual-branch output projections (GPT-2 style)
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]   # (B, 1, T, hd/2)

    def _chunk_of(self, pos):
        c = self.cfg
        return (pos // c.chunk_size).clamp(max=c.n_mem_chunks - 1)

    def _mem_ctx_train(self, T, device):
        pos = torch.arange(T, device=device)[None]                  # (1, T): no padding in training
        chunk = self._chunk_of(pos)
        w, r, has = memory_masks(chunk, chunk, torch.ones_like(pos, dtype=torch.bool), self.cfg.mem_per_chunk, self.n_slots)
        return {"w": w, "r": r, "has": has}

    def trunk(self, idx):
        """(B, T) ids -> (B, T, D) final hidden states"""
        B, T = idx.shape
        cos, sin = self._rope(T)
        x = self.embed(idx)
        mem, mctx = None, None
        for blk in self.blocks:
            if blk.kind == "memory":
                if mem is None:
                    mem = self.mem_init[None].expand(B, -1, -1)       # the learned latent memory enters the first memory layer
                    mctx = self._mem_ctx_train(T, idx.device)
                x, mem = blk(x, mem, mctx)
            else:
                x = blk(x, cos, sin)
        return self.final_norm(x)

    def head_input(self, h, k):
        """hidden state as seen by head k (0-based: head 0 predicts the next token)"""
        if k == 0:
            return h
        return self.mtp_norm[k - 1](h + self.mtp_proj[k - 1](h))

    def forward(self, idx):
        """training / validation path of the next-token head: idx (B, T) -> logits (B, T, V)"""
        return F.linear(self.head_input(self.trunk(idx), 0), self.embed.weight)      # weight tying: logits = h @ E^T

    def mtp_loss(self, w):
        """w: (B, T+1) window. Position t of head k (0-based) is trained on token w[t+1+k]. Returns the weighted mean loss and
        the per-head losses (detached). With mtp_pos_frac < 1 heads 2..K only use a random fraction of the positions."""
        cfg = self.cfg
        x = w[:, :-1]
        T = x.size(1)
        h = self.trunk(x)
        E = self.embed.weight
        recompute = self.training and CE_RECOMPUTE
        total = h.new_zeros((), dtype=torch.float32)
        parts = []
        for k in range(cfg.n_mtp):
            n = T - k
            hk, tgt = h[:, :n], w[:, 1 + k:1 + k + n]
            if k > 0 and self.training and cfg.mtp_pos_frac < 1.0:
                sel = torch.randperm(n, device=h.device)[:max(1, int(n * cfg.mtp_pos_frac))]
                hk, tgt = hk[:, sel], tgt[:, sel]
            hk = self.head_input(hk, k)
            s, c = ce_sum_count(hk.reshape(-1, hk.size(-1)), E, tgt.reshape(-1), cfg.pad_id, CE_CHUNK, ckpt=recompute)
            lk = s / c.clamp(min=1)
            parts.append(lk.detach())
            total = total + cfg.mtp_weights[k] * lk
        return total / sum(cfg.mtp_weights), parts

    # ---- KV-cached generation (next-token head only) ----
    def init_cache(self, B, total_len, dtype, device):
        """one {"k", "v"} per block: the RoPE'd keys/values of a local layer, the (un-rotated) WRITE keys/values of a memory layer.
        cache[0] also carries "pos" (B, total_len): the real position of every slot, -1 = padding / not filled yet."""
        kv, hd = self.cfg.n_kv_heads, self.cfg.d_model // self.cfg.n_heads
        cache = [{"k": torch.zeros(B, kv, total_len, hd, dtype=dtype, device=device),
                  "v": torch.zeros(B, kv, total_len, hd, dtype=dtype, device=device)}
                 for _ in self.blocks]
        cache[0]["pos"] = torch.full((B, total_len), -1, dtype=torch.long, device=device)
        return cache

    def forward_cached(self, idx, pos_ids, cache, start, attn_mask):
        """idx (B, T) new tokens with explicit real positions pos_ids (B, T); returns logits of the last one.
        attn_mask (B, 1, T, start+T) is built by start_generation / next_logits; its LAST query row says which of the
        slots [0, start+T) hold real tokens (the last query is always a real token), which is how padding slots are recognised."""
        B, T = idx.shape
        S = start + T
        c = self.cfg
        cos, sin = self._rope(T, pos_ids)
        pos_store = cache[0]["pos"]
        new_real = attn_mask[:, 0, -1, start:S]
        pos_store[:, start:S] = torch.where(new_real, pos_ids, torch.full_like(pos_ids, -1))
        kpos = pos_store[:, :S]                                      # (B, S) real position of every cached slot, -1 = padding
        x = self.embed(idx)
        mem, mctx = None, None
        for blk, ca in zip(self.blocks, cache):
            if blk.kind == "memory":
                if mem is None:
                    mem = self.mem_init[None].expand(B, -1, -1)
                    w, r, has = memory_masks(self._chunk_of(pos_ids), self._chunk_of(kpos.clamp(min=0)), kpos >= 0,
                                             c.mem_per_chunk, self.n_slots)
                    mctx = {"w": w, "r": r, "has": has}
                x, mem = blk.forward_cached(x, mem, ca, start, mctx)
            else:
                x = blk.forward_cached(x, cos, sin, ca, start, attn_mask, pos_ids, kpos)
        x = self.final_norm(x)
        return F.linear(self.head_input(x[:, -1], 0), self.embed.weight)


# ============================================================================
# 3. PEFT MODULES   teacher: LoRA (additive)   student: LoRMA-plus (multiplicative)   [+ "lora" / "full" for the student]
# ============================================================================
class LoRALinear(nn.Module):
    """y = W x + (alpha / r) * B(A(dropout(x)));  W frozen, A: (r, in) kaiming-init, B: (out, r) zero-init
    -> at step 0 the adapted model is EXACTLY the base model."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        assert base.bias is None, "the Phase-1 models use bias-free projections"
        self.base = base
        self.r = r
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    @property
    def weight(self):                      # lets code that peeks at .weight keep working
        return self.base.weight

    def forward(self, x):
        out = self.base(x)
        if self.merged:
            return out
        lora = F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)), self.lora_B.to(x.dtype))
        return out + lora * self.scaling

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += (self.lora_B @ self.lora_A) * self.scaling
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight -= (self.lora_B @ self.lora_A) * self.scaling
            self.merged = False


class LoRMALinear(nn.Module):
    """Multiplicative low-rank adapter around a FROZEN bias-free Linear W0 (d_out x d_in)  (LoRMA+):
           effective weight  W = M W0 ,  M = I + s B A   (B: d_out x r = 0 at init, A: r x d_out)
       evaluated as  y = W0 x ;  out = y + s * B(A(y))   -> costs about as much as LoRA, and at step 0 it is EXACTLY the base layer."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        assert base.bias is None, "the Phase-1 models use bias-free projections"
        d = base.out_features
        self.base = base
        self.r = r
        self.scaling = alpha / r
        self.lorma_A = nn.Parameter(torch.empty(r, d))
        self.lorma_B = nn.Parameter(torch.zeros(d, r))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False                    # only used as a bypass switch by the self-check ("behave as the plain layer")

    @property
    def weight(self):
        return self.base.weight

    def forward(self, x):
        y = self.base(x)
        if self.merged:
            return y
        z = F.linear(F.linear(self.drop(y), self.lorma_A.to(y.dtype)), self.lorma_B.to(y.dtype))
        return y + z * self.scaling


def wrap_linears(lm, make, targets):
    """replace every nn.Linear whose attribute name is in `targets` (anywhere inside the blocks) by make(linear)"""
    n = 0
    for blk in lm.blocks:
        for parent in list(blk.modules()):
            for name, child in list(parent.named_children()):
                if name in targets and isinstance(child, nn.Linear):
                    setattr(parent, name, make(child))
                    n += 1
    assert n > 0, "no Linear layer matched the PEFT targets"
    return n


def is_adapter_param(name):
    return "lora_" in name or "lorma_" in name


def setup_teacher_peft(lm):
    for p in lm.parameters():
        p.requires_grad = False
    if T_TRAIN_HEAD0:
        for p in lm.heads[0].parameters():
            p.requires_grad = True
    n = wrap_linears(lm, lambda b: LoRALinear(b, T_LORA_R, T_LORA_ALPHA, T_LORA_DROPOUT), T_TARGETS)
    n_ad = sum(p.numel() for nm, p in lm.named_parameters() if is_adapter_param(nm))
    log(f"teacher PEFT: LoRA r={T_LORA_R} alpha={T_LORA_ALPHA} on {n} linear layers ({n_ad / 1e6:.2f}M parameters); "
        f"MTP head 0 trainable: {T_TRAIN_HEAD0}")


def setup_student_peft(lm):
    for p in lm.parameters():
        p.requires_grad = False
    if STUDENT_PEFT == "full":
        for blk in lm.blocks:
            for p in blk.parameters():
                p.requires_grad = True
        lm.mem_init.requires_grad = True
        n, n_ad = 0, 0
    else:
        make = {"lorma": lambda b: LoRMALinear(b, S_R, S_ALPHA, S_DROPOUT),
                "lora": lambda b: LoRALinear(b, S_R, 2 * S_R, S_DROPOUT)}[STUDENT_PEFT]
        n = wrap_linears(lm, make, S_TARGETS)
        n_ad = sum(p.numel() for nm, p in lm.named_parameters() if is_adapter_param(nm))
    if S_TRAIN_FINAL_NORM:
        lm.final_norm.weight.requires_grad = True
    log(f"student PEFT: {STUDENT_PEFT}" + (f" r={S_R} alpha={S_ALPHA} on {n} linear layers ({n_ad / 1e6:.2f}M parameters)" if n else
                                           " (all blocks + memory init trainable, embeddings frozen)"))


# ============================================================================
# 4. TOKENIZER (Phase-1 word-level BPE, copied VERBATIM from the Model 3 / Model 4 notebooks) + DATA -> TOKEN WINDOWS
# ============================================================================


WORD_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+"""
)


class LegalBPE:
    def __init__(self, vocab_path, merges_path, meta_path=None):
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)

        # Load merge rules
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)

         # Tokenizer fingerprint for window-cache validation
        h = hashlib.md5()

        for path in (vocab_path, merges_path, meta_path):
            if path and os.path.exists(path):
                with open(path, "rb") as f:
                    h.update(f.read())

        self.fingerprint = h.hexdigest()[:12]

        
        # Merge rank: lower rank = earlier BPE merge
        self.ranks = {
            (a, b): i
            for i, (a, b) in enumerate(merges)
        }

        # ID -> token
        self.inv = {
            int(i): t
            for t, i in self.vocab.items()
        }

        self.vocab_size = max(self.inv) + 1

        assert len(self.inv) == self.vocab_size, \
            "vocab ids are not contiguous"

        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        self.bos_id = self.eos_id
        self.mask_id = self.eos_id

        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }
        # Your tokenizer metadata
        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

        for key, val in (
            ("pad_id", self.pad_id),
            ("unk_id", self.unk_id),
            ("eos_id", self.eos_id),
            ("bos_id", self.bos_id),
            ("mask_id", self.mask_id),
        ):
            if key in meta and meta[key] != val:
                raise ValueError(
                    f"meta.json {key}={meta[key]} disagrees with vocab ({val})"
            )

        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(
                f"warning: meta.json vocab_size="
                f"{meta['vocab_size']} but vocab has "
                f"{self.vocab_size}"
            )

        # Decoder-only model
        self.bos_id = self.eos_id
        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }

        self.is_end = [
            self.inv[i].endswith("</w>")
            for i in range(self.vocab_size)
        ]

        # Cache BPE results for repeated words
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit

        # Character sequence + word-end marker
        syms = list(word) + ["</w>"]

        # Apply BPE merges
        while len(syms) > 1:
            best = None
            best_rank = None

            for i in range(len(syms) - 1):
                r = self.ranks.get(
                    (syms[i], syms[i + 1])
                )

                if r is not None and (
                    best_rank is None or r < best_rank
                ):
                    best = (
                        syms[i],
                        syms[i + 1]
                    )
                    best_rank = r

            if best is None:
                break

            merged = best[0] + best[1]

            out = []
            i = 0

            while i < len(syms):
                if (
                    i < len(syms) - 1
                    and syms[i] == best[0]
                    and syms[i + 1] == best[1]
                ):
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1

            syms = out

        ids = [
            self.vocab.get(s, self.unk_id)
            for s in syms
        ]

        # Prevent unlimited cache growth
        if len(self.cache) > 3_000_000:
            self.cache.clear()

        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []

        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))

        return ids

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

    def decode(self, ids):
        toks = [
            self.inv.get(int(i), "")
            for i in ids
            if int(i) not in self.special_ids
        ]

        return "".join(toks).replace("</w>", " ").strip()


def load_tokenizer():
    """vocab_50k.json + merges_50k.json (+ meta_50k.json): BPE_DIR, else the first folder under /kaggle/input that has them"""
    dirs = []
    if BPE_DIR:
        dirs.append(BPE_DIR)
    for root in (INPUT_ROOT, os.getcwd(), WORK_DIR):
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "vocab_50k.json" in files and "merges_50k.json" in files:
                    dirs.append(d)
    for d in dirs:
        vp, mp, meta = (os.path.join(d, n) for n in ("vocab_50k.json", "merges_50k.json", "meta_50k.json"))
        if not (os.path.exists(vp) and os.path.exists(mp)):
            continue
        tok = LegalBPE(vp, mp, meta)
        log(f"tokenizer: {d} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id}")
        return tok
    raise FileNotFoundError("vocab_50k.json + merges_50k.json not found: attach the BPE dataset (output of bpe-50k.ipynb) "
                            "or set BPE_DIR=/path/to/folder")


def tokenize_documents(docs, tok):
    """per document: list of token-id lists, one per sentence, each cut to MAX_SENT_TOK (never empty).
    NOTE: the Phase-1 tokenizer drops punctuation, so a sentence made only of punctuation has no tokens -> [unk]."""
    out = []
    for d in docs:
        sents = []
        for s in d["text"]:
            ids = tok.encode(SENT_PREFIX + s.strip())[:MAX_SENT_TOK]
            if SENT_END == "eos" and ids:
                ids = ids[:MAX_SENT_TOK - 1] + [tok.eos_id]
            sents.append(ids if ids else [tok.unk_id])
        d["sent_ids"] = sents
        out.append(sents)
    return out


def make_windows(sent_ids, rng=None):
    """
    Cover EVERY sentence exactly once as a *target*.  Each window = [left-context sentences][target block],
    total tokens <= MAX_LEN.  Returns a list of (ctx_start, tgt_start, tgt_end) sentence ranges
    (ctx_start <= tgt_start < tgt_end, half-open).
    rng=None -> deterministic (inference / dev);  rng given -> random target-block size (training augmentation).
    """
    n = len(sent_ids)
    lens = [len(s) for s in sent_ids]
    wins, i = [], 0
    while i < n:
        budget = TARGET_TOK if rng is None else rng.randint(*TRAIN_TARGET_RANGE)
        budget = min(budget, MAX_LEN)
        j, used = i, 0
        while j < n and (j == i or used + lens[j] <= budget):
            used += lens[j]
            j += 1
        # left context: as many previous sentences as still fit
        c, room = i, MAX_LEN - used
        while c > 0 and lens[c - 1] <= room:
            c -= 1
            room -= lens[c]
        wins.append((c, i, j))
        i = j
    return wins


def window_example(doc, label_ids, win, domain_id, soft=None):
    c, a, b = win
    ids, ends, starts = [], [], []
    for k in range(c, b):
        starts.append(len(ids))
        ids.extend(doc["sent_ids"][k])
        ends.append(len(ids) - 1)
    n = len(doc["sent_ids"])
    return {
        "ids": ids,
        "ends": ends[a - c:],
        "starts": starts[a - c:],
        "labels": [label_ids[k] for k in range(a, b)] if label_ids is not None else [-100] * (b - a),
        "pos": [min(int(k / n * N_POS_BUCKETS), N_POS_BUCKETS - 1) for k in range(a, b)],
        "domain": domain_id,
        "sent_idx": list(range(a, b)),
        "soft": soft[a:b] if soft is not None else None,      # teacher logits (n_targets, K) when distilling
    }


def collate(examples, pad_id, device):
    B = len(examples)
    T = max(len(e["ids"]) for e in examples)
    S = max(len(e["ends"]) for e in examples)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    ends = torch.zeros(B, S, dtype=torch.long)
    starts = torch.zeros(B, S, dtype=torch.long)
    labels = torch.full((B, S), -100, dtype=torch.long)
    pos = torch.zeros(B, S, dtype=torch.long)
    smask = torch.zeros(B, S, dtype=torch.bool)
    dom = torch.zeros(B, dtype=torch.long)
    has_soft = examples[0].get("soft") is not None
    soft = torch.zeros(B, S, int(examples[0]["soft"].shape[-1])) if has_soft else None
    for r, e in enumerate(examples):
        ids[r, :len(e["ids"])] = torch.tensor(e["ids"])           # right padding is safe: attention is causal
        m = len(e["ends"])
        ends[r, :m] = torch.tensor(e["ends"])
        starts[r, :m] = torch.tensor(e["starts"])
        labels[r, :m] = torch.tensor(e["labels"])
        pos[r, :m] = torch.tensor(e["pos"])
        smask[r, :m] = True
        dom[r] = e["domain"]
        if has_soft:
            soft[r, :m] = torch.from_numpy(np.asarray(e["soft"], dtype=np.float32))
    out = dict(ids=ids, ends=ends, starts=starts, labels=labels, pos=pos, smask=smask, dom=dom)
    if has_soft:
        out["soft"] = soft
    return {k: v.to(device) for k, v in out.items()}


def build_examples(docs, l2i, domain2id, rng=None, soft=None):
    """soft: optional {doc_id: (n_sentences, K) teacher logits} attached to every window's target sentences"""
    out = []
    for d in docs:
        label_ids = [l2i[x] for x in d["labels"]] if d.get("labels") is not None else None
        sd = soft[d["id"]] if soft is not None else None
        for w in make_windows(d["sent_ids"], rng):
            out.append(window_example(d, label_ids, w, domain2id[d["domain"]], sd))
    return out


# ============================================================================
# 5. DATA REPORT  (everything later steps need, in one json)
# ============================================================================
def compute_class_weights(counts):
    """counts: (n_domains, K) train label counts -> (n_domains, K) capped inverse-sqrt weights.
    Labels with ZERO examples in a domain get weight 0 there (they can never be a target in that domain)."""
    D, K = counts.shape
    pooled = counts.sum(0)
    W = np.zeros((D, K), dtype=np.float32)
    for d in range(D):
        c = counts[d] if PER_DOMAIN_WEIGHTS else pooled
        present = c > 0
        N, Kd = c.sum(), present.sum()
        W[d, present] = np.clip((N / (Kd * c[present])) ** CLASS_WEIGHT_POWER, *CLASS_WEIGHT_CLIP)
    return W


def rare_mask_from_counts(counts):
    """(D, K) bool: label is present in the domain's train set but below RARE_FRAC of its sentences"""
    share = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    return (counts > 0) & (share < RARE_FRAC)


def data_report(data, labels, l2i, tok):
    rep = {"labels": labels, "num_labels": len(labels), "splits": {}}
    for name, docs in data.items():
        tokenize_documents(docs, tok)
        sent_tok = np.array([len(s) for d in docs for s in d["sent_ids"]])
        raw_tok = [len(tok.encode(SENT_PREFIX + s.strip())) for d in docs for s in d["text"]]
        n_sent = [len(d["text"]) for d in docs]
        entry = {
            "documents": len(docs), "sentences": int(sum(n_sent)),
            "sentences_per_doc": {"mean": float(np.mean(n_sent)), "median": float(np.median(n_sent)),
                                  "max": int(max(n_sent))},
            "tokens_per_sentence": {"mean": float(np.mean(raw_tok)), "p95": float(np.percentile(raw_tok, 95)),
                                    "p99": float(np.percentile(raw_tok, 99)), "max": int(max(raw_tok))},
            f"sentences_truncated_at_{MAX_SENT_TOK}_tokens": int((np.array(raw_tok) > MAX_SENT_TOK).sum()),
            "tokens_per_doc_mean": float(sent_tok.sum() / len(docs)),
            "windows_per_epoch_inference_style": int(sum(len(make_windows(d["sent_ids"])) for d in docs)),
        }
        if name.endswith("_train") or name.endswith("_dev"):
            cnt = Counter(x for d in docs for x in d["labels"])
            entry["label_counts"] = {l: cnt.get(l, 0) for l in labels}
        rep["splits"][name] = entry

    # per-domain label counts from the TRAIN splits (one shared model, but weights/priors are domain specific)
    counts = np.zeros((len(DOMAINS), len(labels)), dtype=np.int64)
    for di, dom in enumerate(DOMAINS):
        c = Counter(x for d in data[f"{dom}_train"] for x in d["labels"])
        counts[di] = [c.get(l, 0) for l in labels]
    W = compute_class_weights(counts)
    rep["train_label_counts"] = {l: int(counts[:, i].sum()) for i, l in enumerate(labels)}
    rep["train_label_counts_by_domain"] = {dom: {l: int(counts[di, i]) for i, l in enumerate(labels)}
                                           for di, dom in enumerate(DOMAINS)}
    rep["class_weights_by_domain"] = {dom: {l: float(W[di, i]) for i, l in enumerate(labels)}
                                      for di, dom in enumerate(DOMAINS)}
    rep["labels_absent_from_train_by_domain"] = {dom: [l for i, l in enumerate(labels) if counts[di, i] == 0]
                                                 for di, dom in enumerate(DOMAINS)}
    rep["window_note"] = (f"MAX_LEN={MAX_LEN}, TARGET_TOK={TARGET_TOK}, MAX_SENT_TOK={MAX_SENT_TOK}")

    log("=" * 90)
    log("DATA REPORT")
    log("=" * 90)
    for name, e in rep["splits"].items():
        log(f"{name:9s} docs={e['documents']:3d} sents={e['sentences']:6d} "
            f"tok/sent mean={e['tokens_per_sentence']['mean']:.1f} p99={e['tokens_per_sentence']['p99']:.0f} "
            f"max={e['tokens_per_sentence']['max']} | truncated={e[f'sentences_truncated_at_{MAX_SENT_TOK}_tokens']} "
            f"| windows/epoch={e['windows_per_epoch_inference_style']}")
    rare = rare_mask_from_counts(counts)
    log("\ntrain label counts and class weights per domain  (R = rare -> windows oversampled):")
    log(f"  {'':24s} " + " | ".join(f"{dom:>6s} count  share  weight" for dom in DOMAINS))
    for i, l in enumerate(labels):
        cells = []
        for di in range(len(DOMAINS)):
            share = 100 * counts[di, i] / max(counts[di].sum(), 1)
            cells.append(f"{counts[di, i]:12d} {share:5.2f}% {W[di, i]:6.2f}{'R' if rare[di, i] else ' '}")
        log(f"  {l:24s} " + " | ".join(cells))
    for di, dom in enumerate(DOMAINS):
        if rep["labels_absent_from_train_by_domain"][dom]:
            log(f"labels with ZERO {dom} train examples:", rep["labels_absent_from_train_by_domain"][dom])
        dev_c = Counter(x for d in data[f"{dom}_dev"] for x in d["labels"])
        missed = {l: n for l, n in dev_c.items() if counts[di, l2i[l]] == 0}
        if missed:
            log(f"WARNING: {dom}_dev contains labels unseen in {dom}_train (cannot be learned in-domain): {missed}")
    return rep, torch.tensor(W, dtype=torch.float32), counts


# ============================================================================
# 6. RR MODEL (either base model) + CHECKPOINT LOADING
# ============================================================================
class RRModel(nn.Module):
    """base model (Model 3 or Model 4, + PEFT) -> sentence vectors (last token [+ mean]) -> 13-way classifier"""

    def __init__(self, lm, kind, n_labels, n_domains=2):
        super().__init__()
        assert kind in ("m3", "m4")
        self.lm, self.kind = lm, kind
        d = lm.cfg.d_model
        d_in = d * (2 if POOLING == "last_mean" else 1)
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, 16) if USE_POS_FEATURE else None
        self.dom_emb = nn.Embedding(n_domains, 8) if USE_DOMAIN_FEATURE else None
        d_in += (16 if USE_POS_FEATURE else 0) + (8 if USE_DOMAIN_FEATURE else 0)
        self.head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(d_in, HEAD_HIDDEN), nn.GELU(),
                                  nn.Dropout(HEAD_DROPOUT), nn.Linear(HEAD_HIDDEN, n_labels))

    def features(self, ids):
        """(B, T) -> (B, T, d): head-0 features of Model 3, final-norm output of Model 4 (both feed the tied LM head)"""
        return self.lm.hidden(ids) if self.kind == "m3" else self.lm.trunk(ids)

    def adapter_parameters(self):
        return [p for n, p in self.lm.named_parameters() if p.requires_grad and is_adapter_param(n)]

    def backbone_parameters(self):
        return [p for n, p in self.lm.named_parameters() if p.requires_grad and not is_adapter_param(n)]

    def head_parameters(self):
        mods = [self.head] + [m for m in (self.pos_emb, self.dom_emb) if m is not None]
        return [p for m in mods for p in m.parameters()]

    def forward(self, b):
        h = self.features(b["ids"])                                           # (B, T, d)
        B, S = b["ends"].shape
        d = h.size(-1)
        last = h.gather(1, b["ends"][:, :, None].expand(B, S, d))             # (B, S, d)
        feats = [last]
        if POOLING == "last_mean":
            csum = torch.cat([torch.zeros(B, 1, d, device=h.device, dtype=h.dtype), h.float().cumsum(1).to(h.dtype)], dim=1)
            hi = csum.gather(1, (b["ends"] + 1)[:, :, None].expand(B, S, d))
            lo = csum.gather(1, b["starts"][:, :, None].expand(B, S, d))
            cnt = (b["ends"] + 1 - b["starts"]).clamp(min=1)[:, :, None].to(h.dtype)
            feats.append((hi - lo) / cnt)
        if self.pos_emb is not None:
            feats.append(self.pos_emb(b["pos"]).to(h.dtype))
        if self.dom_emb is not None:
            feats.append(self.dom_emb(b["dom"])[:, None, :].expand(B, S, -1).to(h.dtype))
        return self.head(torch.cat(feats, dim=-1))                            # (B, S, n_labels)


# ---- trainable-only checkpoints (adapter + head: a few MB) ----
def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in keep}


def save_adapter(model, path, extra):
    torch.save({"state": trainable_state(model), "kind": model.kind, "pooling": POOLING, **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


def load_adapter(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    assert ck["kind"] == model.kind, f"{path} is a {ck['kind']} checkpoint, model is {model.kind}"
    res = model.load_state_dict(ck["state"], strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return ck


# ---- find + load the pretrained base checkpoints (same logic as phase3-model3.ipynb, for any tag) ----
def read_json_safe(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def find_ckpts(tag, explicit=""):
    """best.pt (lowest val loss) before latest.pt, tag-checked through the meta json"""
    if explicit:
        return [explicit] if os.path.exists(explicit) else []
    cands = []
    for root in (INPUT_ROOT, "/kaggle/working", os.getcwd()):
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            for name, meta_name in (("best.pt", "best_meta.json"), ("latest.pt", "latest_meta.json")):
                if name not in files:
                    continue
                meta = read_json_safe(os.path.join(d, meta_name)) or {}
                if meta.get("tag") not in (None, tag):
                    continue
                rank = (0 if name == "best.pt" else 1, meta.get("val_loss", meta.get("best_val", 1e9)))
                cands.append((rank, os.path.join(d, name)))
    cands.sort(key=lambda c: c[0])
    return [p for _, p in cands]


def pick_ckpt(tag, explicit, what):
    paths = find_ckpts(tag, explicit)
    if not paths:
        raise FileNotFoundError(f"no {what} checkpoint found (best.pt/latest.pt with tag '{tag}'). Attach the dataset that contains "
                                f"its checkpoints folder, or set the env var {'MODEL3_CKPT' if what == 'teacher' else 'MODEL4_CKPT'}")
    for p in paths:
        c = torch.load(p, map_location="cpu", weights_only=False)
        if c.get("tag") not in (None, tag):
            log(f"skip {p}: tag {c.get('tag')!r} != {tag!r}")
            continue
        log(f"{what} checkpoint: {p} | val_loss={c.get('val_loss')} epoch={c.get('epoch')} step={c.get('global_step')}")
        return c, p
    raise FileNotFoundError(f"{what}: checkpoints found but none carries tag '{tag}': {paths}")


def clean_state(sd):
    return {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}


def load_teacher_lm(tok, device):
    ck, _ = pick_ckpt(TEACHER_TAG, TEACHER_CKPT_PATH, "teacher")
    cfg = config_from_ckpt3(ck)
    if cfg.real_vocab != tok.vocab_size:
        raise ValueError(f"Model 3 real_vocab={cfg.real_vocab} but the tokenizer has {tok.vocab_size} entries")
    cfg.dropout = T_FT_DROPOUT
    lm = DecoderLM3(cfg)
    sd = {k: v for k, v in clean_state(ck["model"]).items() if not (k.startswith("heads.") and not k.startswith("heads.0."))}
    lm.load_state_dict(sd, strict=True)
    log(f"Model 3 (teacher): d_model={cfg.d_model} layers={cfg.n_layers} heads={cfg.n_heads}/{cfg.n_kv_heads} ffn={cfg.ffn_hidden} "
        f"max_len={cfg.max_len} | {sum(p.numel() for p in lm.parameters()) / 1e6:.1f}M parameters, loaded strictly")
    setup_teacher_peft(lm)
    return lm.to(device), ck


def load_student_lm(tok, device):
    ck, _ = pick_ckpt(STUDENT_TAG, STUDENT_CKPT_PATH, "student")
    cfg = config_from_ckpt4(ck)
    if cfg.vocab_size < tok.vocab_size:
        raise ValueError(f"Model 4 has {cfg.vocab_size} embedding rows but the tokenizer has {tok.vocab_size} entries")
    if cfg.vocab_size != tok.vocab_size:
        log(f"note: Model 4 embedding rows {cfg.vocab_size} != tokenizer vocab {tok.vocab_size}")
    assert MAX_LEN <= cfg.max_len, f"MAX_LEN {MAX_LEN} > Model 4 max_len {cfg.max_len}"
    cfg.ffn_dropout = S_FT_DROPOUT
    lm = DecoderLM4(cfg)
    lm.load_state_dict(clean_state(ck["model"]), strict=True)
    log(f"Model 4 (student): d_model={cfg.d_model} layers={cfg.n_layers} {list(cfg.layer_types)} heads={cfg.n_heads}/{cfg.n_kv_heads} "
        f"ffn={cfg.ffn_hidden} | {sum(p.numel() for p in lm.parameters()) / 1e6:.1f}M parameters, loaded strictly")
    setup_student_peft(lm)
    return lm.to(device), ck


@torch.no_grad()
def lm_probe_loss(kind, lm, tok, docs, device, n_tok=512, skip=16):
    """next-token CE of the PRETRAINED model (tied embedding) on Phase-3 text: close to the checkpoint's val_loss when the
    tokenizer is the right one, ~ln(vocab) ~ 10.8 when it is not"""
    text = " ".join(s for d in docs[:3] for s in d["text"][:400])
    ids = tok.encode(text)[:n_tok + 1]
    if len(ids) < skip + 8:
        return float("nan")
    lm.eval()
    x = torch.tensor(ids[:-1], dtype=torch.long, device=device)[None]
    y = torch.tensor(ids[1:], dtype=torch.long, device=device)
    z = lm.hidden(x) if kind == "m3" else lm.trunk(x)
    logits = F.linear(z, lm.embed.weight[:tok.vocab_size])[0].float()
    return F.cross_entropy(logits, y, reduction="none")[skip:].mean().item()


def check_tokenizer_matches(kind, lm, tok, docs, ck, device):
    loss = lm_probe_loss(kind, lm, tok, docs, device)
    ref = ck.get("val_loss")
    log(f"tokenizer probe [{kind}]: pretrained next-token loss on Phase-3 text = {loss:.3f} (checkpoint's own val_loss = {ref})")
    bad = loss != loss or loss > 7.5 or (ref is not None and loss > ref + 3.5)
    if bad:
        msg = (f"the pretrained {kind} model does not reproduce its language-model loss with this tokenizer ({loss:.2f} vs val_loss {ref}); "
               "the checkpoint was probably trained with different BPE files. (FORCE_TOKENIZER=1 turns this into a warning.)")
        if not FORCE_TOKENIZER:
            raise RuntimeError(msg)
        log("WARNING: " + msg)


# ============================================================================
# 7a. EVAL / PREDICT / LOSS HELPERS  (as in phase3-model3)
# ============================================================================
def macro_prf(y_true, y_pred, labels):
    """macro P/R/F1 over labels that occur in gold OR prediction (sklearn default); also per-class table"""
    from sklearn.metrics import precision_recall_fscore_support
    present = sorted(set(y_true) | set(y_pred))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=present, average=None, zero_division=0)
    per = {labels[c]: dict(p=float(p[i]), r=float(r[i]), f1=float(f[i]), support=int(s[i])) for i, c in enumerate(present)}
    return dict(macro_p=float(np.mean(p)), macro_r=float(np.mean(r)), macro_f1=float(np.mean(f)), per_class=per)

def predict_docs(model, docs, domain2id, pad_id, device, eval_bs=8, return_logits=False):
    """-> {doc_id: [label_id per sentence]} (and optionally {doc_id: (n, K) logits})"""
    model.eval()
    preds, logit_store = {}, {}
    for d in docs:
        n = len(d["sent_ids"])
        exs = [window_example(d, None, w, domain2id[d["domain"]]) for w in make_windows(d["sent_ids"])]
        out = np.full(n, -1, dtype=np.int64)
        lg = None
        for i in range(0, len(exs), eval_bs):
            chunk = exs[i:i + eval_bs]
            b = collate(chunk, pad_id, device)
            with amp_ctx():
                logits = model(b).float()
            if lg is None:
                lg = np.zeros((n, logits.size(-1)), dtype=np.float32)
            for r, e in enumerate(chunk):
                m = len(e["sent_idx"])
                lr = logits[r, :m].cpu().numpy()
                lg[e["sent_idx"]] = lr
                out[e["sent_idx"]] = lr.argmax(-1)
        assert (out >= 0).all() and len(out) == n, f"{d['id']}: not every sentence was predicted"
        preds[d["id"]] = out.tolist()
        logit_store[d["id"]] = lg
    return (preds, logit_store) if return_logits else preds

def evaluate_dev(model, dev_docs, labels, l2i, domain2id, pad_id, device):
    preds = predict_docs(model, dev_docs, domain2id, pad_id, device)
    res = {}
    for name, sel in (("ALL", dev_docs), ("CL", [d for d in dev_docs if d["domain"] == "CL"]),
                      ("IT", [d for d in dev_docs if d["domain"] == "IT"])):
        if not sel:
            continue
        yt = [l2i[x] for d in sel for x in d["labels"]]
        yp = [p for d in sel for p in preds[d["id"]]]
        res[name] = macro_prf(yt, yp, labels)
        res[name]["acc"] = float(np.mean(np.array(yt) == np.array(yp)))
    return res

def weighted_ce(logits, b, class_w):
    """class_w: (n_domains, K).  Each target token is weighted by ITS domain's weight for ITS label;
    normalised by the summed weight of the micro-batch (a weighted mean, so the loss scale stays ~constant)."""
    K = logits.size(-1)
    y = b["labels"]
    valid = y >= 0
    ce = F.cross_entropy(logits.float().reshape(-1, K), y.reshape(-1), ignore_index=-100,
                         reduction="none", label_smoothing=LABEL_SMOOTHING).view_as(y)
    w = class_w[b["dom"][:, None].expand_as(y), y.clamp(min=0)] * valid
    return (ce * w).sum() / w.sum().clamp(min=1e-6)

def oversample_rare(exs, rare_mask):
    out = []
    for e in exs:
        out.append(e)
        if any(rare_mask[e["domain"], y] for y in e["labels"] if y >= 0):
            out.extend([e] * (RARE_REPEAT - 1))
    return out

def make_epoch_examples(train_docs, l2i, domain2id, rng, rare_mask, soft=None):
    exs = build_examples(train_docs, l2i, domain2id, rng, soft)      # fresh random window boundaries
    return oversample_rare(exs, rare_mask) if OVERSAMPLE_RARE else exs

def build_post(counts):
    """per-domain log-priors (for logit adjustment) and the set of labels seen in each domain's train split"""
    allowed = counts > 0
    prior = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    logp = np.log(np.where(allowed, prior, 1.0))
    for d in range(len(counts)):                    # unseen label: no boost (treated like the most frequent label)
        logp[d, ~allowed[d]] = logp[d, allowed[d]].max()
    return {"log_prior": logp.astype(np.float32), "allowed": allowed, "tau": 0.0, "use_mask": False}

def decode_logits(logit_store, docs, domain2id, post, tau=None, use_mask=None):
    """argmax of  logits - tau * log(prior_domain)   (tau=0: plain argmax), optionally forbidding labels unseen in-domain"""
    tau = post["tau"] if tau is None else tau
    use_mask = post["use_mask"] if use_mask is None else use_mask
    preds = {}
    for d in docs:
        di = domain2id[d["domain"]]
        lg = logit_store[d["id"]].copy()
        if tau:
            lg -= tau * post["log_prior"][di][None, :]
        if use_mask:
            lg[:, ~post["allowed"][di]] = -1e9
        preds[d["id"]] = lg.argmax(-1).tolist()
    return preds

def lr_scale(step, total, warmup):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    prog = (step - warmup) / max(1, total - warmup)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

def write_submission(preds_by_id, data, labels):
    expected = {d["id"]: len(d["text"]) for k in ("CL_test", "IT_test") for d in data[k]}
    assert set(preds_by_id) == set(expected), "prediction ids differ from the test ids"
    out, valid = {}, set(labels)
    for doc_id, n in expected.items():                       # test-file order, CL then IT
        lab = [labels[i] for i in preds_by_id[doc_id]]
        assert len(lab) == n and all(l in valid for l in lab), doc_id
        out[doc_id] = lab
    os.makedirs(WORK_DIR, exist_ok=True)
    pred_path, zip_path = os.path.join(WORK_DIR, "prediction.json"), os.path.join(WORK_DIR, "submission.zip")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    dist = Counter(l for v in out.values() for l in v)
    log(f"wrote {pred_path} and {zip_path} | {len(out)} docs, {sum(len(v) for v in out.values())} sentences")
    log("predicted label distribution:", dict(dist.most_common()))


# ============================================================================
# 7b. KD LOSS, TRAINING LOOP, CALIBRATION
# ============================================================================
def kd_loss(logits, b, class_w, alpha, temp):
    """sum_i w_i [ alpha CE(s_i, y_i) + (1-alpha) T^2 KL(softmax(t_i/T) || softmax(s_i/T)) ] / sum_i w_i   (i = target sentences)
    w_i = capped inverse-sqrt class weight of the gold label of sentence i in its domain; context sentences have label -100 -> w=0"""
    K = logits.size(-1)
    y = b["labels"]
    valid = y >= 0
    ce = F.cross_entropy(logits.float().reshape(-1, K), y.reshape(-1), ignore_index=-100,
                         reduction="none", label_smoothing=LABEL_SMOOTHING).view_as(y)
    t = F.log_softmax(b["soft"].float() / temp, dim=-1)
    s = F.log_softmax(logits.float() / temp, dim=-1)
    kl = (t.exp() * (t - s)).sum(-1)                                      # (B, S)
    w = class_w[b["dom"][:, None].expand_as(y), y.clamp(min=0)] * valid
    return ((alpha * ce + (1.0 - alpha) * temp * temp * kl) * w).sum() / w.sum().clamp(min=1e-6)


def train(model, tag, train_docs, dev_docs, labels, l2i, domain2id, class_w, tok, device, rare_mask,
          epochs, lr_adapter, lr_backbone, soft=None, alpha=1.0, temp=1.0):
    """soft=None -> gold-label class-weighted CE (teacher);  soft={doc_id: (n, K) teacher logits} -> distillation loss (student).
    Early stopping on dev Macro-F1; best trainable weights -> WORK_DIR/<tag>_best.pt.  returns (best_f1, best_epoch)"""
    os.makedirs(CKPT_DIR, exist_ok=True)
    groups = [("adapter", model.adapter_parameters(), lr_adapter, 0.0),
              ("backbone", model.backbone_parameters(), lr_backbone, 0.0),
              ("head", model.head_parameters(), LR_HEAD, WEIGHT_DECAY)]
    groups = [g for g in groups if g[1]]
    all_params = [p for g in groups for p in g[1]]
    n_train = sum(p.numel() for p in all_params)
    n_base = sum(p.numel() for n, p in model.lm.named_parameters() if not is_adapter_param(n))
    log(f"[{tag}] trainable: " + " + ".join(f"{nm} {sum(p.numel() for p in ps) / 1e6:.3f}M" for nm, ps, _, _ in groups)
        + f" = {n_train / 1e6:.2f}M ({100 * n_train / (n_base + n_train):.1f}% of all parameters; base model {n_base / 1e6:.1f}M)")
    log(f"[{tag}] loss: " + (f"gold CE only" if soft is None else f"KD  alpha={alpha} T={temp}  (gold CE + soft-label KL, class-weighted)"))

    opt = torch.optim.AdamW([{"params": ps, "lr": lr, "base_lr": lr, "weight_decay": wd} for _, ps, lr, wd in groups],
                            betas=(0.9, 0.999), eps=1e-8)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP["enabled"] and AMP["dtype"] == torch.float16)
    class_w = class_w.to(device)

    rng = random.Random(SEED)
    steps_per_epoch = max(1, math.ceil(len(make_epoch_examples(train_docs, l2i, domain2id, random.Random(SEED), rare_mask, soft))
                                       / (MICRO_BS * ACCUM)))
    total_steps, warmup = epochs * steps_per_epoch, int(WARMUP_FRAC * epochs * steps_per_epoch)
    log(f"[{tag}] epochs={epochs} steps/epoch~{steps_per_epoch} total~{total_steps} effective batch={MICRO_BS * ACCUM} windows")

    best_f1, best_epoch, bad, gstep = -1.0, -1, 0, 0
    log_path = os.path.join(CKPT_DIR, f"train_log_{tag}.csv")
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "dev_macro_f1", "dev_cl_f1", "dev_it_f1", "dev_acc", "sec"])
    best_path = os.path.join(WORK_DIR, f"{tag}_best.pt")

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        exs = make_epoch_examples(train_docs, l2i, domain2id, rng, rare_mask, soft)    # fresh windows + rare-label oversampling
        rng.shuffle(exs)
        run, nrun = 0.0, 0
        for i in range(0, len(exs), MICRO_BS * ACCUM):
            group = exs[i:i + MICRO_BS * ACCUM]
            n_micro = math.ceil(len(group) / MICRO_BS)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_scale(gstep, total_steps, warmup)
            for j in range(0, len(group), MICRO_BS):
                b = collate(group[j:j + MICRO_BS], tok.pad_id, device)
                with amp_ctx():
                    logits = model(b)
                loss = weighted_ce(logits, b, class_w) if soft is None else kd_loss(logits, b, class_w, alpha, temp)
                scaler.scale(loss / n_micro).backward()
                run += loss.item()
                nrun += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            gstep += 1

        res = evaluate_dev(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device)
        f1 = res["ALL"]["macro_f1"]
        row = [epoch + 1, f"{run / max(nrun, 1):.4f}", f"{f1:.4f}", f"{res.get('CL', {}).get('macro_f1', float('nan')):.4f}",
               f"{res.get('IT', {}).get('macro_f1', float('nan')):.4f}", f"{res['ALL']['acc']:.4f}", f"{time.time() - t0:.0f}"]
        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow(row)
        improved = f1 > best_f1
        log(f"[{tag} epoch {epoch + 1:2d}/{epochs}] loss {row[1]} | dev macro-F1 {row[2]} (CL {row[3]} / IT {row[4]}) "
            f"acc {row[5]} | {row[6]}s {'<-- best' if improved else ''}")
        if improved:
            best_f1, best_epoch, bad = f1, epoch + 1, 0
            save_adapter(model, best_path, {"epoch": epoch + 1, "dev_macro_f1": f1})
        else:
            bad += 1
            if bad >= PATIENCE:
                log(f"[{tag}] early stop: no dev improvement for {PATIENCE} epochs")
                break
    log(f"[{tag}] best dev macro-F1 {best_f1:.4f} at epoch {best_epoch}")
    return best_f1, best_epoch


def calibrate_from_store(store, dev_docs, labels, l2i, domain2id, counts, verbose=False):
    """tau (prior-correction strength) + unseen-label mask tuned on dev Macro-F1 for a ready-made {doc_id: (n, K)} store"""
    post = build_post(counts)
    yt = [l2i[x] for d in dev_docs for x in d["labels"]]
    best = None
    for use_mask in (False, True):
        for tau in TAU_GRID:
            pr = decode_logits(store, dev_docs, domain2id, post, tau, use_mask)
            f1 = macro_prf(yt, [p for d in dev_docs for p in pr[d["id"]]], labels)["macro_f1"]
            if verbose:
                log(f"  calibration: tau={tau:.2f} mask_unseen={use_mask!s:5s} -> dev macro-F1 {f1:.4f}")
            if best is None or f1 > best[0] + 1e-4:          # ties go to the simpler setting (tried first)
                best = (f1, tau, use_mask)
    post["tau"], post["use_mask"] = best[1], best[2]
    return post, best[0]


def dev_f1(store, dev_docs, labels, l2i, domain2id, post):
    pr = decode_logits(store, dev_docs, domain2id, post)
    return macro_prf([l2i[x] for d in dev_docs for x in d["labels"]], [p for d in dev_docs for p in pr[d["id"]]], labels)


def log_softmax_store(store):
    out = {}
    for k, v in store.items():
        m = v.max(-1, keepdims=True)
        out[k] = v - m - np.log(np.exp(v - m).sum(-1, keepdims=True))
    return out


def avg_stores(a, b):
    return {k: 0.5 * (a[k] + b[k]) for k in a}


# ============================================================================
# 8. SELF-CHECKS
# ============================================================================
def run_self_checks(model, device):
    """1. [Model 4] block-wise sliding-window attention == masked reference (else fall back to the masked version)
       2. a freshly wrapped PEFT model is an exact no-op (LoRA / LoRMA bypass == adapted forward)
       3. causality + right padding: changing tokens from position p on never moves earlier features (also across memory chunks)
       4. the RR forward runs on a hand-made batch; gradients reach the adapter + head and never the frozen embedding"""
    model.eval()
    lm = model.lm
    saved, AMP["enabled"] = AMP["enabled"], False
    try:
        if model.kind == "m4":
            g = torch.Generator().manual_seed(0)
            worst = 0.0
            for window in (128, 256):
                for T in (window - 5, 300, 640):
                    q, k, v = (torch.randn(2, 4, T, 16, generator=g).to(device) for _ in range(3))
                    worst = max(worst, (local_attention(q, k, v, window) - local_attention_ref(q, k, v, window)).abs().max().item())
            if worst > 1e-4:
                LOCAL_IMPL["mode"] = "mask"
                log(f"WARNING: block-local attention differs from the masked reference by {worst:.2e} -> using masked SDPA")
            else:
                log(f"self-check [{model.kind}]: block-local attention == masked reference (max diff {worst:.1e})")
        V = lm.embed.weight.size(0)
        with torch.no_grad():
            x = torch.randint(5, V, (1, 700), device=device)
            f1 = model.features(x)
            scale = max(1.0, f1.abs().mean().item())
            wraps = [m for m in lm.modules() if isinstance(m, (LoRALinear, LoRMALinear))]
            noop = 0.0
            if wraps:
                for m in wraps:
                    m.merged = True                        # bypass: plain pretrained layers
                base_out = model.features(x)
                for m in wraps:
                    m.merged = False
                noop = (f1 - base_out).abs().max().item()
                assert noop < 1e-4 * scale, f"PEFT is not a no-op at init: max|diff|={noop:.2e}"
            x2 = x.clone()
            x2[:, 400:] = torch.randint(5, V, (1, 300), device=device)
            past = (f1[:, :400] - model.features(x2)[:, :400]).abs().max().item()
            assert past < 1e-3 * scale, f"causality broken: changing tokens >= 400 moved earlier features by {past:.2e}"

        K = model.head[-1].out_features
        b = {"ids": torch.randint(5, V, (2, 40), device=device),
             "ends": torch.tensor([[9, 19, 39], [14, 29, 39]], device=device),
             "starts": torch.tensor([[0, 10, 20], [0, 15, 30]], device=device),
             "pos": torch.zeros(2, 3, dtype=torch.long, device=device),
             "dom": torch.tensor([0, 1], device=device)}
        with torch.enable_grad():
            out = model(b)
            assert out.shape == (2, 3, K) and torch.isfinite(out).all(), "RR forward failed"
            out.float().pow(2).sum().backward()

        def got(ps):
            ps = list(ps)
            return (not ps) or any(p.grad is not None and p.grad.abs().sum() > 0 for p in ps)
        ok_ad, ok_head = got(model.adapter_parameters()), got(model.head_parameters())
        embed_grad = lm.embed.weight.grad is not None
        model.zero_grad(set_to_none=True)
        assert ok_ad and ok_head, f"no gradient reached adapter={ok_ad} head={ok_head}"
        assert not embed_grad, "gradient reached the frozen embedding"
    finally:
        AMP["enabled"] = saved
    log(f"self-checks passed [{model.kind}]: PEFT no-op at init (max|diff| {noop:.1e}), causal / right-padding safe "
        f"(past change {past:.1e}), RR forward OK, gradients reach adapter + head only")


# ============================================================================
# 9. STAGES + MAIN
# ============================================================================
def find_file(name):
    """name in WORK_DIR, else anywhere under /kaggle/input (an earlier session's output attached as input)"""
    roots = [WORK_DIR] if SMOKE else [WORK_DIR, INPUT_ROOT, os.getcwd()]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if name in files:
                return os.path.join(d, name)
    return None


def soft_is_valid(soft, train_docs, labels):
    if soft.get("labels") != labels:
        return False
    return all(d["id"] in soft["train"] and soft["train"][d["id"]].shape == (len(d["text"]), len(labels)) for d in train_docs)


def stage_teacher(ctx):
    """A. fine-tune the teacher (Model 3 + LoRA) unless teacher_soft.pt already exists; returns the soft-label stores"""
    labels, train_docs = ctx["labels"], ctx["train_docs"]
    path = find_file("teacher_soft.pt")
    if path:
        soft = torch.load(path, map_location="cpu", weights_only=False)
        if soft_is_valid(soft, train_docs, labels):
            log(f"stage A skipped: teacher soft labels found in {path} (teacher dev macro-F1 {soft.get('teacher_dev_macro_f1')})")
            return soft
        log(f"{path} does not match the current data/labels -> recomputing")

    device, tok, domain2id = ctx["device"], ctx["tok"], ctx["domain2id"]
    lm, ck = load_teacher_lm(tok, device)
    check_tokenizer_matches("m3", lm, tok, ctx["data"]["CL_train"] + ctx["data"]["IT_train"], ck, device)
    model = RRModel(lm, "m3", len(labels)).to(device)
    run_self_checks(model, device)

    adapter = find_file("teacher_best.pt")
    if adapter:
        log(f"teacher adapter found ({adapter}) -> training skipped")
        load_adapter(model, adapter)
    else:
        train(model, "teacher", train_docs, ctx["dev_docs"], labels, ctx["l2i"], domain2id, ctx["class_w"], tok, device,
              ctx["rare_mask"], ctx["epochs_t"], LR_T_ADAPTER, LR_T_BACKBONE)
        load_adapter(model, os.path.join(WORK_DIR, "teacher_best.pt"))
    res = evaluate_dev(model, ctx["dev_docs"], labels, ctx["l2i"], domain2id, tok.pad_id, device)
    log(f"\nTEACHER dev: macro-P {res['ALL']['macro_p']:.4f} macro-R {res['ALL']['macro_r']:.4f} macro-F1 {res['ALL']['macro_f1']:.4f}")

    log("teacher soft labels for train / dev / test ...")
    soft = {"labels": labels, "teacher_dev_macro_f1": res["ALL"]["macro_f1"]}
    for key, docs in (("train", train_docs), ("dev", ctx["dev_docs"]), ("test", ctx["test_docs"])):
        _, soft[key] = predict_docs(model, docs, domain2id, tok.pad_id, device, return_logits=True)
    torch.save(soft, os.path.join(WORK_DIR, "teacher_soft.pt"))
    del model, lm
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return soft


def stage_student(ctx, soft):
    """B. distil into Model 4 (+ LoRMA); returns {"dev": store, "test": store} of the best student"""
    labels, device, tok, domain2id = ctx["labels"], ctx["device"], ctx["tok"], ctx["domain2id"]
    lm, ck = load_student_lm(tok, device)
    check_tokenizer_matches("m4", lm, tok, ctx["data"]["CL_train"] + ctx["data"]["IT_train"], ck, device)
    model = RRModel(lm, "m4", len(labels)).to(device)
    run_self_checks(model, device)
    lr_ad = LR_S_BACKBONE if STUDENT_PEFT == "full" else LR_S_ADAPTER
    train(model, "student", ctx["train_docs"], ctx["dev_docs"], labels, ctx["l2i"], domain2id, ctx["class_w"], tok, device,
          ctx["rare_mask"], ctx["epochs_s"], lr_ad, LR_S_BACKBONE, soft=soft["train"], alpha=KD_ALPHA, temp=KD_TEMP)
    ck = load_adapter(model, os.path.join(WORK_DIR, "student_best.pt"))
    res = evaluate_dev(model, ctx["dev_docs"], labels, ctx["l2i"], domain2id, tok.pad_id, device)
    log(f"\nSTUDENT (best epoch {ck['epoch']}) dev: macro-P {res['ALL']['macro_p']:.4f} macro-R {res['ALL']['macro_r']:.4f} "
        f"macro-F1 {res['ALL']['macro_f1']:.4f}")
    for l, v in res["ALL"]["per_class"].items():
        log(f"  {l:24s} P {v['p']:.3f} R {v['r']:.3f} F1 {v['f1']:.3f} support {v['support']}")
    with open(os.path.join(WORK_DIR, "student_dev_report.json"), "w") as f:
        json.dump(res, f, indent=2)
    out = {}
    for key, docs in (("dev", ctx["dev_docs"]), ("test", ctx["test_docs"])):
        _, out[key] = predict_docs(model, docs, domain2id, tok.pad_id, device, return_logits=True)
    return out


def dump_predictions(preds, data, labels, path):
    out = {d["id"]: [labels[i] for i in preds[d["id"]]] for k in ("CL_test", "IT_test") for d in data[k]}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)


def main():
    set_seed()
    os.makedirs(WORK_DIR, exist_ok=True)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible - training will be very slow")

    data, labels, l2i = load_phase3()
    tok = load_tokenizer()
    domain2id = {d: i for i, d in enumerate(DOMAINS)}

    report, class_w, counts = data_report(data, labels, l2i, tok)       # tokenises every split with the shared tokenizer
    rare_mask = rare_mask_from_counts(counts)
    with open(os.path.join(WORK_DIR, "data_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    train_docs = data["CL_train"] + data["IT_train"]
    dev_docs = data["CL_dev"] + data["IT_dev"]
    test_docs = data["CL_test"] + data["IT_test"]
    epochs_t, epochs_s = EPOCHS_TEACHER, EPOCHS_STUDENT
    if SMOKE:
        train_docs = data["CL_train"][:2] + data["IT_train"][:2]
        dev_docs = data["CL_dev"][:1] + data["IT_dev"][:1]
        test_docs = data["CL_test"][:1] + data["IT_test"][:1]
        epochs_t = epochs_s = 1
        log("SMOKE=1: 2+2 train docs, 1+1 dev docs, 1+1 test docs, 1 epoch per stage, no submission written")
    ids = [d["id"] for d in train_docs + dev_docs + test_docs]
    assert len(ids) == len(set(ids)), "document ids are not unique across splits (soft labels are keyed by id)"

    ctx = dict(data=data, labels=labels, l2i=l2i, tok=tok, device=device, domain2id=domain2id, class_w=class_w,
               rare_mask=rare_mask, train_docs=train_docs, dev_docs=dev_docs, test_docs=test_docs,
               epochs_t=epochs_t, epochs_s=epochs_s)

    soft = stage_teacher(ctx)                       # A
    stu = stage_student(ctx, soft)                  # B

    # ---------------- C. report: teacher / student / ensemble ----------------
    log("\n" + "=" * 90 + "\nKD RESULT (dev Macro-F1; calibration = tau + unseen-label mask tuned on dev, per model)\n" + "=" * 90)
    stores = {"teacher": {"dev": soft["dev"], "test": soft["test"]}, "student": stu}
    stores["ensemble"] = {k: avg_stores(log_softmax_store(soft[k]), log_softmax_store(stu[k])) for k in ("dev", "test")}
    posts, summary = {}, {}
    for name, st in stores.items():
        raw = dev_f1(st["dev"], dev_docs, labels, l2i, domain2id, build_post(counts))
        if POSTHOC_CALIBRATION:
            posts[name], _ = calibrate_from_store(st["dev"], dev_docs, labels, l2i, domain2id, counts, verbose=(name == SUBMIT_MODEL))
        else:
            posts[name] = build_post(counts)
        cal = dev_f1(st["dev"], dev_docs, labels, l2i, domain2id, posts[name])
        summary[name] = {"raw_macro_f1": raw["macro_f1"], "calibrated_macro_f1": cal["macro_f1"], "calibrated_macro_p": cal["macro_p"],
                         "calibrated_macro_r": cal["macro_r"], "tau": posts[name]["tau"], "mask_unseen": bool(posts[name]["use_mask"])}
        log(f"  {name:9s} raw {raw['macro_f1']:.4f} | calibrated macro-F1 {cal['macro_f1']:.4f} (P {cal['macro_p']:.4f} R {cal['macro_r']:.4f}) "
            f"tau={posts[name]['tau']} mask_unseen={posts[name]['use_mask']}")
    summary["kd"] = {"alpha": KD_ALPHA, "temperature": KD_TEMP, "student_peft": STUDENT_PEFT, "submit_model": SUBMIT_MODEL}
    with open(os.path.join(WORK_DIR, "kd_report.json"), "w") as f:
        json.dump(summary, f, indent=2)

    preds = {name: decode_logits(st["test"], test_docs, domain2id, posts[name]) for name, st in stores.items()}
    if SMOKE:
        log(f"SMOKE: predicted {sum(len(v) for v in preds[SUBMIT_MODEL].values())} test sentences (submission not written)")
    else:
        for name in preds:
            dump_predictions(preds[name], data, labels, os.path.join(WORK_DIR, f"prediction_{name}.json"))
        log(f"\nsubmission = {SUBMIT_MODEL} predictions")
        write_submission(preds[SUBMIT_MODEL], data, labels)
    log("done.")


if __name__ == "__main__":
    main()
